# Klasifikasi Penyakit Daun Padi Menggunakan ResNet-50 *Transfer Learning* dan Interpretasi Grad-CAM

Notebook final penelitian skripsi. Urutan bagian mengikuti alur penelitian (Bab III):

1. Persiapan lingkungan
2. Memuat dataset
3. Eksplorasi data
4. Konfigurasi *hyperparameter*
5. Pembagian data, *preprocessing*, dan augmentasi
6. Arsitektur model ResNet-50 (*Transfer Learning*)
7. Pelatihan tahap 1: kepala klasifikasi (*backbone* dibekukan)
8. Pelatihan tahap 2: *fine-tuning*
9. Grafik akurasi dan *loss*
10. Evaluasi pada data uji
11. Visualisasi Grad-CAM
12. Analisis fokus Grad-CAM
13. Validasi *5-fold cross validation*
14. Penyimpanan model
15. Prediksi citra baru
16. Ringkasan hasil

**Cara menjalankan:** *Runtime → Change runtime type → GPU (T4)*, lalu *Runtime → Run all*.
Semua gambar dan tabel hasil disimpan otomatis ke folder Google Drive `PROJECT/hasil_skripsi/` untuk keperluan Bab IV.

## 1. Persiapan Lingkungan
Memuat seluruh pustaka, memeriksa GPU, dan menetapkan *seed* agar hasil dapat direproduksi.

In [ ]:
import os, sys, json, time, gc, random, shutil, zipfile, hashlib, collections
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy import ndimage as ndi
from matplotlib.colors import rgb_to_hsv

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             recall_score, classification_report, confusion_matrix)

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU tersedia:", gpus if gpus else "TIDAK ADA - aktifkan lewat Runtime > Change runtime type")

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

## 2. Memuat Dataset
Dataset citra daun padi (Kaggle) disimpan di Google Drive dengan struktur satu subfolder per kelas:

```
Daun Padi/
├── blast/    (200 citra)
├── blight/   (220 citra)
└── tungro/   (80 citra)
```

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SRC_PATH = '/content/drive/MyDrive/Daun Padi'               # lokasi dataset (folder atau .zip) di Drive
    OUT_DIR  = '/content/drive/MyDrive/PROJECT/hasil_skripsi'   # semua hasil disimpan di sini
else:
    SRC_PATH = os.environ.get('SRC_PATH', 'Daun Padi')
    OUT_DIR  = os.environ.get('OUT_DIR', 'hasil_skripsi')

assert os.path.exists(SRC_PATH), "Dataset tidak ditemukan di Drive. Cek nama dan lokasinya."
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists('dataset_raw'):
    if os.path.isdir(SRC_PATH):
        shutil.copytree(SRC_PATH, 'dataset_raw/Daun Padi')
    else:
        with zipfile.ZipFile(SRC_PATH) as z:
            z.extractall('dataset_raw')

IMG_EXT = {'.jpg', '.jpeg', '.png'}

def find_data_dir(root='dataset_raw'):
    """Cari folder yang berisi subfolder-subfolder kelas berisi gambar."""
    root = Path(root)
    for c in [root] + [p for p in root.rglob('*') if p.is_dir()]:
        if c.name.startswith('__MACOSX'):
            continue
        subdirs = [d for d in c.iterdir() if d.is_dir() and not d.name.startswith(('.', '__'))]
        if len(subdirs) >= 2 and any(any(f.suffix.lower() in IMG_EXT for f in sd.iterdir() if f.is_file())
                                     for sd in subdirs):
            return c
    return None

DATA_DIR = find_data_dir()
assert DATA_DIR is not None, "Dataset tidak ditemukan. Pastikan struktur folder benar."
class_names = sorted(d.name for d in DATA_DIR.iterdir()
                     if d.is_dir() and not d.name.startswith(('.', '__'))
                     and any(f.suffix.lower() in IMG_EXT for f in d.iterdir() if f.is_file()))
K = len(class_names)
print("DATA_DIR :", DATA_DIR)
print("Kelas    :", class_names)
print("Output   :", OUT_DIR)

## 3. Eksplorasi Data
Menghitung jumlah citra per kelas, memeriksa variasi ukuran citra dan duplikat, serta menampilkan contoh citra.

In [ ]:
def list_images(cname):
    return sorted(f for f in (DATA_DIR / cname).iterdir()
                  if f.is_file() and f.suffix.lower() in IMG_EXT and not f.name.startswith('._'))

rows, sizes, hashes = [], collections.Counter(), collections.defaultdict(list)
for cname in class_names:
    files = list_images(cname)
    rows.append({'kelas': cname, 'jumlah': len(files)})
    for f in files:
        with Image.open(f) as im:
            sizes[im.size] += 1
        hashes[hashlib.md5(f.read_bytes()).hexdigest()].append(f'{cname}/{f.name}')

df_count = pd.DataFrame(rows)
df_count['persentase (%)'] = (df_count['jumlah'] / df_count['jumlah'].sum() * 100).round(2)
display(df_count)
df_count.to_csv(f'{OUT_DIR}/tabel_jumlah_data.csv', index=False)

print("Ukuran citra (lebar, tinggi) terbanyak:", sizes.most_common(6))
dups = {h: fs for h, fs in hashes.items() if len(fs) > 1}
print("Kelompok citra duplikat:", len(dups))

plt.figure(figsize=(5, 3.5))
plt.bar(df_count['kelas'], df_count['jumlah'], color=['#4C72B0', '#55A868', '#C44E52'])
for i, v in enumerate(df_count['jumlah']):
    plt.text(i, v + 3, str(v), ha='center')
plt.title('Distribusi Jumlah Citra per Kelas'); plt.ylabel('Jumlah citra')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/distribusi_kelas.png', dpi=200); plt.show()

fig, axes = plt.subplots(K, 6, figsize=(15, 2.7 * K))
for r, cname in enumerate(class_names):
    for c, f in enumerate(list_images(cname)[:6]):
        axes[r, c].imshow(Image.open(f).convert('RGB')); axes[r, c].axis('off')
        if c == 0: axes[r, c].set_title(cname, loc='left', fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/contoh_citra.png', dpi=150); plt.show()

## 4. Konfigurasi *Hyperparameter*

In [ ]:
IMG_SIZE        = 224    # ukuran input ResNet-50
BATCH_SIZE      = 32
EPOCHS_HEAD     = 15     # tahap 1: latih kepala klasifikasi (maksimum, dengan EarlyStopping)
EPOCHS_FINETUNE = 15     # tahap 2: fine-tuning (maksimum, dengan EarlyStopping)
FINE_TUNE_AT    = 30     # jumlah layer teratas ResNet-50 yang dibuka saat fine-tuning
LR_HEAD         = 1e-3   # learning rate tahap 1 (Adam)
LR_FINETUNE     = 1e-5   # learning rate tahap 2 (Adam)
DROPOUT         = 0.3
DENSE_UNITS     = 256
USE_CLASS_WEIGHT = True  # menyeimbangkan pengaruh kelas tungro yang jumlahnya sedikit
N_SPLITS        = 5      # jumlah fold validasi silang

config = {k: v for k, v in list(globals().items()) if k.isupper() and isinstance(v, (int, float, bool))}
json.dump(config, open(f'{OUT_DIR}/konfigurasi.json', 'w'), indent=2)
pd.Series(config)

## 5. Pembagian Data, *Preprocessing*, dan Augmentasi
- Data dibagi **per file** secara *stratified* 70:15:15 (latih/validasi/uji) agar proporsi kelas sama dan tidak ada kebocoran data.
- *Preprocessing*: *resize* 224×224 piksel dan normalisasi `preprocess_input` ResNet-50 (sesuai bobot ImageNet).
- Augmentasi hanya pada data latih: *flip* horizontal, rotasi, *zoom*, dan kontras.

In [ ]:
paths, labels = [], []
for idx, cname in enumerate(class_names):
    for f in list_images(cname):
        paths.append(str(f)); labels.append(idx)
paths, labels = np.array(paths), np.array(labels)
print("Total citra:", len(paths))

p_trainval, p_test, y_trainval, y_test = train_test_split(
    paths, labels, test_size=0.15, stratify=labels, random_state=SEED)
p_train, p_val, y_train, y_val = train_test_split(
    p_trainval, y_trainval, test_size=0.15 / 0.85, stratify=y_trainval, random_state=SEED)

assert not (set(p_train) & set(p_val)) and not (set(p_train) & set(p_test)) and not (set(p_val) & set(p_test)), \
    "KEBOCORAN: ada file di lebih dari satu subset!"

split_rows = []
for name, y in [('Latih', y_train), ('Validasi', y_val), ('Uji', y_test)]:
    row = {'subset': name, 'total': len(y)}
    row.update({c: int((y == i).sum()) for i, c in enumerate(class_names)})
    split_rows.append(row)
df_split = pd.DataFrame(split_rows)
display(df_split)
df_split.to_csv(f'{OUT_DIR}/tabel_pembagian_data.csv', index=False)

def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return img, label

def make_augment():
    return tf.keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.1),
    ], name='augmentasi')

def make_ds(p, y, augment=None, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((p, y))
    if shuffle:
        ds = ds.shuffle(len(p), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_img, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    if augment is not None:
        ds = ds.map(lambda x, t: (preprocess_input(augment(x, training=True)), t), num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(lambda x, t: (preprocess_input(x), t), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

augment = make_augment()
train_ds = make_ds(p_train, y_train, augment=augment, shuffle=True)
val_ds   = make_ds(p_val, y_val)
test_ds  = make_ds(p_test, y_test)

def get_class_weight(y):
    if not USE_CLASS_WEIGHT:
        return None
    # Hanya hitung bobot untuk kelas yang benar-benar ada di data
    active_classes = np.unique(y)
    w = compute_class_weight('balanced', classes=active_classes, y=y)
    return {int(i): float(v) for i, v in zip(active_classes, w)}

class_weight = get_class_weight(y_train)
print("Class weight:", {class_names[i]: round(v, 3) for i, v in class_weight.items()} if class_weight else None)

# Contoh hasil augmentasi (untuk Bab III/IV)
sample, _ = load_img(p_train[0], 0)
plt.figure(figsize=(12, 3))
plt.subplot(1, 5, 1); plt.imshow(sample.numpy().astype('uint8')); plt.title('Asli'); plt.axis('off')
for i in range(4):
    aug_img = augment(sample[None, ...], training=True)[0].numpy()
    plt.subplot(1, 5, i + 2); plt.imshow(np.clip(aug_img, 0, 255).astype('uint8')); plt.title(f'Augmentasi {i+1}'); plt.axis('off')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/contoh_augmentasi.png', dpi=150); plt.show()

## 6. Arsitektur Model ResNet-50 (*Transfer Learning*)
ResNet-50 dengan bobot ImageNet digunakan sebagai ekstraktor fitur (tanpa lapisan klasifikasi asli), kemudian ditambahkan kepala klasifikasi:
*Global Average Pooling* → *Dropout* 0,3 → *Dense* 256 (ReLU) → *Dropout* 0,3 → *Dense* 3 (*softmax*).

In [ ]:
def build_model():
    base = ResNet50(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_citra')
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(DROPOUT, name='dropout_1')(x)
    x = layers.Dense(DENSE_UNITS, activation='relu', name='dense_relu')(x)
    x = layers.Dropout(DROPOUT, name='dropout_2')(x)
    outputs = layers.Dense(K, activation='softmax', name='output_softmax')(x)
    return models.Model(inputs, outputs, name='ResNet50_TL'), base

def unfreeze_top(base, n):
    """Buka n layer teratas ResNet-50; BatchNormalization tetap beku."""
    base.trainable = True
    for l in base.layers[:-n]:
        l.trainable = False
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False

def new_callbacks():
    return [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)]

def compile_model(m, lr):
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

def predict_all(m, ds):
    yt, yp = [], []
    for x, t in ds:
        yp.extend(np.argmax(m.predict(x, verbose=0), axis=1)); yt.extend(t.numpy())
    return np.array(yt), np.array(yp)

tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
model, base = build_model()
model.summary()

## 7. Pelatihan Tahap 1 — Kepala Klasifikasi (*backbone* dibekukan)
Seluruh layer ResNet-50 dibekukan; hanya kepala klasifikasi yang dilatih dengan Adam (*learning rate* 0,001).

In [ ]:
compile_model(model, LR_HEAD)
t0 = time.time()
hist_head = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                      callbacks=new_callbacks(), class_weight=class_weight)
time_head = (time.time() - t0) / 60

best_val_head = float(np.min(hist_head.history['val_loss']))
STAGE1_W = 'bobot_tahap1.weights.h5'
model.save_weights(STAGE1_W)               # simpan bobot terbaik tahap 1
yt, yp = predict_all(model, test_ds)
acc_test_head = accuracy_score(yt, yp)
print(f"Tahap 1 selesai: {len(hist_head.history['loss'])} epoch | val_loss terbaik = {best_val_head:.4f} | "
      f"akurasi uji = {acc_test_head:.4f} | {time_head:.1f} menit")

## 8. Pelatihan Tahap 2 — *Fine-tuning*
30 layer teratas ResNet-50 dibuka (BatchNormalization tetap beku) dan dilatih ulang dengan Adam (*learning rate* 0,00001).
Model akhir dipilih berdasarkan **val_loss terbaik** dari kedua tahap, sehingga *fine-tuning* hanya dipakai jika benar-benar memperbaiki performa pada data validasi.

In [ ]:
unfreeze_top(base, FINE_TUNE_AT)
compile_model(model, LR_FINETUNE)
print("Parameter dapat dilatih:", sum(int(np.prod(w.shape)) for w in model.trainable_weights))

t0 = time.time()
hist_ft = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
                    callbacks=new_callbacks(), class_weight=class_weight)
time_ft = (time.time() - t0) / 60
best_val_ft = float(np.min(hist_ft.history['val_loss']))

if best_val_ft <= best_val_head:
    final_stage = 'Tahap 2 (fine-tuning)'
else:
    model.load_weights(STAGE1_W)
    final_stage = 'Tahap 1 (kepala klasifikasi)'
print(f"val_loss terbaik tahap 1 = {best_val_head:.4f} | tahap 2 = {best_val_ft:.4f}")
print("Model akhir yang dipakai:", final_stage)

## 9. Grafik Akurasi dan *Loss*
Garis putus-putus menandai batas antara tahap 1 dan tahap 2.

In [ ]:
def concat(h1, h2, key):
    return list(h1.history.get(key, [])) + list(h2.history.get(key, []))

acc, val_acc = concat(hist_head, hist_ft, 'accuracy'), concat(hist_head, hist_ft, 'val_accuracy')
loss, val_loss = concat(hist_head, hist_ft, 'loss'), concat(hist_head, hist_ft, 'val_loss')
ep = np.arange(1, len(acc) + 1); b = len(hist_head.history['accuracy'])

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.plot(ep, acc, 'o-', label='Latih'); plt.plot(ep, val_acc, 's-', label='Validasi')
plt.axvline(b + 0.5, ls='--', c='gray'); plt.title('Akurasi'); plt.xlabel('Epoch'); plt.legend()
plt.subplot(1, 2, 2); plt.plot(ep, loss, 'o-', label='Latih'); plt.plot(ep, val_loss, 's-', label='Validasi')
plt.axvline(b + 0.5, ls='--', c='gray'); plt.title('Loss'); plt.xlabel('Epoch'); plt.legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/grafik_akurasi_loss.png', dpi=200); plt.show()

## 10. Evaluasi pada Data Uji
*Accuracy*, *Precision*, *Recall*, *F1-Score*, dan *Confusion Matrix*.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
y_true, y_pred = predict_all(model, test_ds)
print(f"Akurasi uji: {test_acc:.4f} | Loss uji: {test_loss:.4f}\n")

# Filter class_names dan K untuk hanya menyertakan kelas yang memiliki data
active_labels = np.unique(y_true)
active_class_names = [class_names[i] for i in active_labels]
active_K = len(active_labels)

print(classification_report(y_true, y_pred, labels=active_labels, target_names=active_class_names, digits=4))

p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=active_labels, zero_division=0)
df_report = pd.DataFrame({'Kelas': active_class_names, 'Precision': p, 'Recall': r, 'F1-Score': f, 'Jumlah': s})
pm, rm, fm, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
df_report.loc[len(df_report)] = ['Rata-rata (macro)', pm, rm, fm, int(s.sum())]
df_report[['Precision', 'Recall', 'F1-Score']] = (df_report[['Precision', 'Recall', 'F1-Score']] * 100).round(2)
display(df_report)
df_report.to_csv(f'{OUT_DIR}/tabel_evaluasi_uji.csv', index=False)

cm = confusion_matrix(y_true, y_pred, labels=active_labels)
plt.figure(figsize=(6, 4.8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=active_class_names, yticklabels=active_class_names, cbar=False,
            annot_kws={'size': 14})
plt.ylabel('Label sebenarnya'); plt.xlabel('Prediksi'); plt.title('Confusion Matrix Data Uji')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/confusion_matrix_uji.png', dpi=200); plt.show()

## 11. Visualisasi Grad-CAM
Grad-CAM dihitung pada layer konvolusi terakhir ResNet-50 (`conv5_block3_out`). Gradien skor kelas terhadap *feature map* dirata-ratakan menjadi bobot tiap kanal, lalu kombinasi berbobot *feature map* dilewatkan ReLU dan dinormalisasi menjadi *heatmap*.

In [ ]:
LAST_CONV = 'conv5_block3_out'
conv_model = tf.keras.Model(base.input, base.get_layer(LAST_CONV).output)
head_layers = [model.get_layer(n) for n in ['gap', 'dropout_1', 'dense_relu', 'dropout_2', 'output_softmax']]

def gradcam(img_batch, class_index=None):
    with tf.GradientTape() as tape:
        conv_out = conv_model(img_batch, training=False)
        tape.watch(conv_out)
        x = conv_out
        for layer in head_layers:
            x = layer(x, training=False)
        preds = x
        if class_index is None:
            class_index = int(tf.argmax(preds[0]))
        score = preds[:, class_index]
    grads = tape.gradient(score, conv_out)                      # (1, 7, 7, 2048)
    weights = tf.reduce_mean(grads, axis=(1, 2))                # bobot alpha tiap kanal
    cam = tf.reduce_sum(conv_out * weights[:, None, None, :], axis=-1)[0]
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE))[..., 0].numpy()
    return cam, class_index, preds.numpy()[0]

def load_one(path):
    """Preprocessing identik dengan pipeline pelatihan."""
    img, _ = load_img(path, 0)
    img = img.numpy()
    arr = preprocess_input(img.copy())[None, ...]
    return np.clip(np.round(img), 0, 255).astype('uint8'), arr

test_info = []
for pth, t in zip(p_test, y_test):
    _, arr = load_one(pth)
    probs = model.predict(arr, verbose=0)[0]
    test_info.append((pth, int(t), int(np.argmax(probs)), float(np.max(probs))))
acc_per_file = np.mean([t == p for _, t, p, _ in test_info])
print(f"Akurasi prediksi per-file: {acc_per_file:.4f} (harus sama dengan akurasi uji {test_acc:.4f})")

def show_gradcam(items, title, fname):
    if not items:
        print(f"{title}: (tidak ada contoh)"); return
    n = len(items)
    fig, axes = plt.subplots(2, n, figsize=(3.2 * n, 6.6), squeeze=False)
    for j, (pth, t, p, conf) in enumerate(items):
        img, arr = load_one(pth)
        cam, _, _ = gradcam(arr, class_index=p)
        axes[0, j].imshow(img); axes[0, j].axis('off'); axes[0, j].set_title(f"asli: {class_names[t]}", fontsize=9)
        axes[1, j].imshow(img); axes[1, j].imshow(cam, cmap='jet', alpha=0.45); axes[1, j].axis('off')
        axes[1, j].set_title(f"prediksi: {class_names[p]} ({conf*100:.0f}%)", fontsize=9)
    fig.suptitle(title, fontsize=12); plt.tight_layout()
    fig.savefig(f'{OUT_DIR}/{fname}.png', dpi=200, bbox_inches='tight'); plt.show()

for ci, cname in enumerate(class_names):
    show_gradcam([x for x in test_info if x[1] == ci and x[2] == ci][:3],
                 f"Grad-CAM — prediksi BENAR: {cname}", f"gradcam_benar_{cname}")
show_gradcam([x for x in test_info if x[1] != x[2]][:8], "Grad-CAM — prediksi SALAH", "gradcam_salah")

## 12. Analisis Fokus Grad-CAM
Mengukur seberapa besar porsi *heatmap* yang jatuh di **latar belakang** (di luar daun). Area daun dideteksi dengan masker warna sederhana (HSV).
- `luas_latar`: proporsi luas latar belakang pada citra
- `panas_di_latar`: proporsi intensitas *heatmap* yang jatuh di latar belakang
- `rasio` = panas_di_latar / luas_latar (> 1 berarti model lebih memperhatikan latar daripada proporsi luasnya)

**Periksa dulu gambar cek masker** sebelum memakai angkanya.

In [ ]:
def leaf_mask(img_uint8):
    hsv = rgb_to_hsv(img_uint8 / 255.0)
    h, s = hsv[..., 0], hsv[..., 1]
    m = (s > 0.20) & (h < 0.50)          # daun: hijau/kuning/cokelat; latar: putih/abu-abu
    m = ndi.binary_opening(m, iterations=2)
    m = ndi.binary_closing(m, iterations=3)
    return ndi.binary_fill_holes(m)

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for j, (pth, t, p, conf) in enumerate(test_info[:6]):
    img, _ = load_one(pth)
    axes[0, j].imshow(img); axes[0, j].axis('off'); axes[0, j].set_title(class_names[t], fontsize=9)
    axes[1, j].imshow(leaf_mask(img), cmap='gray'); axes[1, j].axis('off')
plt.suptitle("Cek masker daun (putih = daun)"); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/cek_masker_daun.png', dpi=150, bbox_inches='tight'); plt.show()

rows = []
for pth, t, p, conf in test_info:
    img, arr = load_one(pth)
    cam, _, _ = gradcam(arr, class_index=p)
    m = leaf_mask(img)
    area_out = 1 - m.mean()
    heat_out = cam[~m].sum() / (cam.sum() + 1e-8)
    rows.append({'kelas': class_names[t], 'benar': t == p, 'luas_latar': area_out,
                 'panas_di_latar': heat_out, 'rasio': heat_out / (area_out + 1e-8)})
df_bg = pd.DataFrame(rows)
cols = ['luas_latar', 'panas_di_latar', 'rasio']
print("Rata-rata seluruh data uji:"); print(df_bg[cols].mean().round(3).to_string())
print("\nPer kelas:");            display(df_bg.groupby('kelas')[cols].mean().round(3))
print("\nBenar vs salah:");       display(df_bg.groupby('benar')[cols].mean().round(3))
df_bg.groupby('kelas')[cols].mean().round(3).to_csv(f'{OUT_DIR}/tabel_fokus_gradcam.csv')

## 13. Validasi *5-Fold Cross Validation*
Untuk memastikan hasil tidak bergantung pada satu pembagian data, seluruh 500 citra dibagi menjadi 5 *fold* (*stratified*).
Pada tiap *fold*, 4 bagian dipakai untuk latih/validasi dan 1 bagian untuk uji, dengan konfigurasi dan aturan pemilihan model **yang sama persis** dengan bagian 7–8.

*Checkpoint* tiap *fold* disimpan ke Drive sehingga jika *runtime* terputus, *fold* yang sudah selesai tidak diulang. Proses ini ±20 menit dengan GPU T4.

In [ ]:
CKPT_DIR = f'{OUT_DIR}/cv_ckpt_3kelas'
os.makedirs(CKPT_DIR, exist_ok=True)
FOLDS = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(paths, labels))

def train_one_fold(fold, tr_idx, te_idx):
    t0 = time.time()
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.utils.set_random_seed(SEED + fold)
    p_tv, y_tv, p_te, y_te = paths[tr_idx], labels[tr_idx], paths[te_idx], labels[te_idx]
    p_tr, p_va, y_tr, y_va = train_test_split(p_tv, y_tv, test_size=0.15, stratify=y_tv, random_state=SEED)
    assert not (set(p_tr) & set(p_va)) and not (set(p_tr) & set(p_te)) and not (set(p_va) & set(p_te))

    cw = get_class_weight(y_tr)
    ds_tr = make_ds(p_tr, y_tr, augment=make_augment(), shuffle=True)
    ds_va, ds_te = make_ds(p_va, y_va), make_ds(p_te, y_te)

    m, b = build_model()
    compile_model(m, LR_HEAD)
    h1 = m.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_HEAD, callbacks=new_callbacks(), class_weight=cw, verbose=0)
    v1 = float(np.min(h1.history['val_loss'])); m.save_weights('cv_tahap1.weights.h5')
    acc_head = float(accuracy_score(*predict_all(m, ds_te)))

    unfreeze_top(b, FINE_TUNE_AT)
    compile_model(m, LR_FINETUNE)
    h2 = m.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_FINETUNE, callbacks=new_callbacks(), class_weight=cw, verbose=0)
    v2 = float(np.min(h2.history['val_loss']))
    if v2 > v1:
        m.load_weights('cv_tahap1.weights.h5')
    yt, yp = predict_all(m, ds_te)
    return {'fold': fold, 'acc_head': acc_head, 'stage': 2 if v2 <= v1 else 1,
            'accuracy': float(accuracy_score(yt, yp)),
            'macro_f1': float(f1_score(yt, yp, average='macro', zero_division=0)),
            'recall': [float(v) for v in recall_score(yt, yp, average=None, labels=list(range(K)), zero_division=0)],
            'minutes': round((time.time() - t0) / 60, 2),
            'y_true': [int(v) for v in yt], 'y_pred': [int(v) for v in yp]}

cv_rows = []
for fold, (tr_idx, te_idx) in enumerate(FOLDS, 1):
    ck = f'{CKPT_DIR}/fold{fold}.json'
    if os.path.exists(ck):
        r = json.load(open(ck)); print(f"fold {fold}: dimuat dari checkpoint")
    else:
        r = train_one_fold(fold, tr_idx, te_idx); json.dump(r, open(ck, 'w'))
    cv_rows.append(r)
    print(f"fold {fold}/{N_SPLITS} | akurasi = {r['accuracy']:.4f} | macro F1 = {r['macro_f1']:.4f} | "
          f"model tahap {r['stage']} | {r['minutes']} menit")

df_cv = pd.DataFrame([{'Fold': r['fold'], 'Accuracy': r['accuracy'], 'Macro F1': r['macro_f1'],
                       **{f'Recall {c}': r['recall'][i] for i, c in enumerate(class_names)}} for r in cv_rows])
mean, std = df_cv.drop(columns='Fold').mean(), df_cv.drop(columns='Fold').std(ddof=1)
df_cv_show = df_cv.copy()
df_cv_show.loc[len(df_cv_show)] = ['Rata-rata', *mean.values]
df_cv_show.loc[len(df_cv_show)] = ['Std', *std.values]
num_cols = df_cv_show.columns[1:]
df_cv_show[num_cols] = (df_cv_show[num_cols].astype(float) * 100).round(2)
display(df_cv_show)
df_cv_show.to_csv(f'{OUT_DIR}/tabel_5fold.csv', index=False)

yt_all = np.concatenate([r['y_true'] for r in cv_rows]); yp_all = np.concatenate([r['y_pred'] for r in cv_rows])

active_labels = np.unique(yt_all)
active_class_names = [class_names[i] for i in active_labels]

print(classification_report(yt_all, yp_all, labels=active_labels, target_names=active_class_names, digits=4))
plt.figure(figsize=(6, 4.8))
sns.heatmap(confusion_matrix(yt_all, yp_all, labels=active_labels), annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=active_class_names, yticklabels=active_class_names, annot_kws={'size': 14})
plt.ylabel('Label sebenarnya'); plt.xlabel('Prediksi'); plt.title('Confusion Matrix Gabungan 5-Fold')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/confusion_matrix_5fold.png', dpi=200); plt.show()

## 14. Penyimpanan Model
Model disimpan ke Google Drive agar tidak hilang saat *runtime* Colab berakhir.

In [ ]:
model.save(f'{OUT_DIR}/resnet50_daun_padi.keras')
json.dump(class_names, open(f'{OUT_DIR}/class_names.json', 'w'))
print("Model tersimpan di:", f'{OUT_DIR}/resnet50_daun_padi.keras')

## 15. Prediksi Citra Baru
Unggah satu citra daun padi; *preprocessing* sama persis dengan data latih, dan Grad-CAM ditampilkan untuk prediksinya.

In [ ]:
if IN_COLAB:
    from google.colab import files
    for fname in files.upload():
        img, arr = load_one(fname)
        cam, idx, probs = gradcam(arr)
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(img); plt.axis('off'); plt.title('Citra masukan')
        plt.subplot(1, 2, 2); plt.imshow(img); plt.imshow(cam, cmap='jet', alpha=0.45); plt.axis('off')
        plt.title(f"Prediksi: {class_names[idx]} ({probs[idx]*100:.1f}%)")
        plt.tight_layout(); plt.show()
        for name, pr in sorted(zip(class_names, probs), key=lambda x: -x[1]):
            print(f"  {name:10s} {pr*100:6.2f}%")

## 16. Ringkasan Hasil
Angka-angka utama untuk Bab IV.

In [ ]:
summary = {
    'Jumlah data (latih/validasi/uji)': f"{len(p_train)}/{len(p_val)}/{len(p_test)}",
    'Epoch tahap 1 (berhenti di)': len(hist_head.history['loss']),
    'Epoch tahap 2 (berhenti di)': len(hist_ft.history['loss']),
    'Model akhir': final_stage,
    'Akurasi uji setelah tahap 1 (%)': round(acc_test_head * 100, 2),
    'Akurasi uji model akhir (%)': round(test_acc * 100, 2),
    'Loss uji': round(test_loss, 4),
    'Macro F1 uji (%)': round(fm * 100, 2),
    'Akurasi 5-fold (%)': f"{mean['Accuracy']*100:.2f} ± {std['Accuracy']*100:.2f}",
    'Macro F1 5-fold (%)': f"{mean['Macro F1']*100:.2f} ± {std['Macro F1']*100:.2f}",
    'Panas Grad-CAM di latar (rata-rata, %)': round(df_bg['panas_di_latar'].mean() * 100, 1),
}
df_summary = pd.DataFrame(summary.items(), columns=['Keterangan', 'Nilai'])
display(df_summary)
df_summary.to_csv(f'{OUT_DIR}/ringkasan_hasil.csv', index=False)
print("Semua hasil tersimpan di:", OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))